In [23]:
import pandas as pd
import json
from sklearn.metrics import classification_report, mean_absolute_error


In [26]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score, cohen_kappa_score


def read_jsonl(path):
    records = []
    skipped = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                skipped += 1
    if skipped > 0:
        print(f"[WARN] Skipped {skipped} malformed JSONL lines in {path}")
    return records


def normalize_gold(gold_path):
    rows = []
    for obj in read_jsonl(gold_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "gold_rating": obj.get("rating"),
                "gold_comment": obj.get("comment"),
            }
        )
    return pd.DataFrame(rows)


def normalize_pred(pred_path):
    rows = []
    for obj in read_jsonl(pred_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "llm_rating": obj.get("rating"),
                "llm_comment": obj.get("comment"),
                "status": obj.get("status", "success"),
            }
        )
    df = pd.DataFrame(rows)
    if "status" in df.columns:
        df = df[df["status"] == "success"].copy()
    return df


def infer_dataset_name(folder_path):
    return Path(folder_path).name


def infer_model_name(pred_file_name):
    name = pred_file_name.replace(".jsonl", "")
    if name.startswith("gemini"):
        return "Gemini 3.1 Pro"
    if name.startswith("qwen"):
        return "Qwen 2.5 7b"
    if name.startswith("claude"):
        return "Claude Sonnet 5"
    if name.startswith("gpt"):
        return "GPT-4o"
    return name


def map_to_3_class(rating):
    # Collapse 5-point intensity into 3-class polarity
    if rating in [-2, -1]:
        return -1
    if rating == 0:
        return 0
    if rating in [1, 2]:
        return 1
    return rating


def evaluate_one_pair(gold_path, pred_path):
    df_gold = normalize_gold(gold_path)
    df_pred = normalize_pred(pred_path)

    df = pd.merge(df_gold, df_pred, on="article_id", how="inner")

    total_merged = len(df)
    df = df.dropna(subset=["gold_rating", "llm_rating"]).copy()
    valid_rows = len(df)

    if valid_rows == 0:
        return None, None

    df["gold_rating"] = df["gold_rating"].astype(int)
    df["llm_rating"] = df["llm_rating"].astype(int)
    df["abs_error"] = (df["gold_rating"] - df["llm_rating"]).abs()

    df["gold_rating_3c"] = df["gold_rating"].apply(map_to_3_class)
    df["llm_rating_3c"] = df["llm_rating"].apply(map_to_3_class)

    labels_5c = [-2, -1, 0, 1, 2]
    labels_3c = [-1, 0, 1]

    mae_5 = round(mean_absolute_error(df["gold_rating"], df["llm_rating"]), 4)
    acc_5 = round(accuracy_score(df["gold_rating"], df["llm_rating"]), 4)
    macro_f1_5 = round(
        f1_score(df["gold_rating"], df["llm_rating"], labels=labels_5c, average="macro", zero_division=0),
        4,
    )

    metrics = {
        "dataset": infer_dataset_name(Path(gold_path).parent),
        "model": infer_model_name(Path(pred_path).name),
        "coverage_vs_gold": round(valid_rows / max(len(df_gold), 1), 4),
        "valid_rows": valid_rows,
        "mae": mae_5,
        "accuracy": acc_5,
        "macro_f1": macro_f1_5,
        "qwk_5": round(
            cohen_kappa_score(df["gold_rating"], df["llm_rating"], weights="quadratic", labels=labels_5c),
            4,
        ),
        "mae_5": mae_5,
        "acc_5": acc_5,
        "macro_f1_5": macro_f1_5,
        "macro_f1_3": round(
            f1_score(df["gold_rating_3c"], df["llm_rating_3c"], labels=labels_3c, average="macro", zero_division=0),
            4,
        ),
        "acc_3": round(accuracy_score(df["gold_rating_3c"], df["llm_rating_3c"]), 4),
    }

    return metrics, df


def run_batch_evaluation(
    base_dir=".",
    summary_csv="evaluation_summary.csv",
    details_dir="evaluation_details",
):
    base = Path(base_dir).resolve()
    out_dir = base / details_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_rows = []
    total_valid_rows = 0

    gold_files = list(base.rglob("gold_standard*.jsonl"))

    for gold_file in gold_files:
        folder = gold_file.parent

        pred_files = [
            p
            for p in folder.glob("*.jsonl")
            if p.name != gold_file.name and not p.name.startswith("gold_standard")
        ]

        for pred_file in pred_files:
            try:
                metrics, merged_df = evaluate_one_pair(gold_file, pred_file)
            except Exception as e:
                print(f"[WARN] Failed to evaluate {pred_file}: {e}")
                continue

            if metrics is None:
                continue

            summary_rows.append(metrics)
            total_valid_rows += metrics["valid_rows"]

            details_name = f"{metrics['dataset']}__{metrics['model']}.csv"
            merged_df.to_csv(out_dir / details_name, index=False, encoding="utf-8-sig")

    summary_df = pd.DataFrame(summary_rows)

    if len(summary_df) == 0:
        print("No valid evaluation pairs found.")
        return pd.DataFrame(), 0

    summary_df = summary_df.sort_values(
        ["dataset", "qwk_5", "macro_f1_3", "mae_5"], ascending=[True, False, False, True]
    ).reset_index(drop=True)
    summary_df.to_csv(base / summary_csv, index=False, encoding="utf-8-sig")

    print("Saved summary:", str(base / summary_csv))
    print("Saved details folder:", str(out_dir))
    print("Total evaluated pairs:", len(summary_df))

    return summary_df, total_valid_rows


# Quick run from this notebook folder:
summary, count = run_batch_evaluation(
    base_dir=".",
    summary_csv="evaluation_summary.csv",
    details_dir="evaluation_details",
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(summary)
print("Total valid rows across all evaluations:", count)


[WARN] Skipped 1 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gold_standard_pahari.jsonl
[WARN] Skipped 2 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gpt4o_pahari.jsonl
[WARN] Skipped 1 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gold_standard_pahari.jsonl
[WARN] Skipped 1 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gold_standard_pahari.jsonl
[WARN] Skipped 1 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gold_standard_pahari.jsonl
[WARN] Skipped 1 malformed JSONL lines in /Users/md.minhajulhaque/Desktop/thesis/demo_pahari/gold_standard_pahari.jsonl
Saved summary: /Users/md.minhajulhaque/Desktop/thesis/evaluation_summary.csv
Saved details folder: /Users/md.minhajulhaque/Desktop/thesis/evaluation_details
Total evaluated pairs: 80


/Users/md.minhajulhaque/Desktop/thesis/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/md.minhajulhaque/Desktop/thesis/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/md.minhajulhaque/Desktop/thesis/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/md.minhajulhaque/Desktop/thesis/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/Users/md.minhajulhaque/Desktop/thesis/.venv/lib/python3.9/site-packages/skl

,dataset,model,coverage_vs_gold,valid_rows,mae,accuracy,macro_f1,qwk_5,mae_5,acc_5,macro_f1_5,macro_f1_3,acc_3
0,apolitical_relgious_hefajot,Qwen 2.5 7b,0.9444,17,1.2941,0.1765,0.0600,0.0000,1.2941,0.1765,0.0600,0.1000,0.1765
1,apolitical_relgious_hefajot,GPT-4o,0.8889,16,0.0000,1.0000,0.2000,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
2,apolitical_relgious_hefajot,Claude Sonnet 5,1.0000,18,0.0000,1.0000,0.2000,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
3,apolitical_relgious_hefajot,Gemini 3.1 Pro,1.0000,18,0.0000,1.0000,0.2000,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
4,apolitical_relgious_iskon,GPT-4o,1.0000,8,0.2500,0.7500,0.3000,0.5000,0.2500,0.7500,0.3000,0.5000,0.7500
5,apolitical_relgious_iskon,Gemini 3.1 Pro,1.0000,8,0.2500,0.7500,0.1714,0.0000,0.2500,0.7500,0.1714,0.2857,0.7500
6,apolitical_relgious_iskon,Claude Sonnet 5,1.0000,8,0.3750,0.6250,0.1538,0.0000,0.3750,0.6250,0.1538,0.2564,0.6250
7,apolitical_relgious_iskon,Qwen 2.5 7b,0.8750,7,1.7143,0.0000,0.0000,0.0000,1.7143,0.0000,0.0000,0.0952,0.1429
8,apolitical_religious_tablig,Claude Sonnet 5,1.0000,9,0.3333,0.6667,0.3818,0.7429,0.3333,0.6667,0.3818,0.9030,0.8889
9,apolitical_religious_tablig,Gemini 3.1 Pro,1.0000,9,0.2222,0.7778,0.3267,0.6087,0.2222,0.7778,0.3267,0.5444,0.7778


Total valid rows across all evaluations: 1223


In [28]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def build_presentation_outputs(
    summary_csv="evaluation_summary.csv",
    reports_dir="evaluation_reports",
):
    summary_path = Path(summary_csv)
    reports_path = Path(reports_dir)
    reports_path.mkdir(parents=True, exist_ok=True)

    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")

    df = pd.read_csv(summary_path)

    # 1) Clean table for slides (includes both 5-class and 3-class perspectives)
    slide_table = df[[
        "dataset", "model", "valid_rows", "coverage_vs_gold",
        "qwk_5", "mae_5", "acc_5", "macro_f1_5", "macro_f1_3", "acc_3"
    ]].sort_values(["dataset", "qwk_5", "macro_f1_3", "mae_5"], ascending=[True, False, False, True])
    slide_table.to_csv(reports_path / "slide_table.csv", index=False, encoding="utf-8-sig")

    # 2) Dataset x Model pivots for quick comparison
    pivot_qwk = df.pivot_table(index="dataset", columns="model", values="qwk_5", aggfunc="mean")
    pivot_f1_3 = df.pivot_table(index="dataset", columns="model", values="macro_f1_3", aggfunc="mean")
    pivot_acc_5 = df.pivot_table(index="dataset", columns="model", values="acc_5", aggfunc="mean")
    pivot_acc_3 = df.pivot_table(index="dataset", columns="model", values="acc_3", aggfunc="mean")

    pivot_qwk.to_csv(reports_path / "qwk_5_pivot.csv", encoding="utf-8-sig")
    pivot_f1_3.to_csv(reports_path / "macro_f1_3_pivot.csv", encoding="utf-8-sig")
    pivot_acc_5.to_csv(reports_path / "acc_5_pivot.csv", encoding="utf-8-sig")
    pivot_acc_3.to_csv(reports_path / "acc_3_pivot.csv", encoding="utf-8-sig")

    # 3) Average score by model across all key metrics
    model_avg = df.groupby("model", as_index=False).agg(
        avg_qwk_5=("qwk_5", "mean"),
        avg_mae_5=("mae_5", "mean"),
        avg_acc_5=("acc_5", "mean"),
        avg_macro_f1_5=("macro_f1_5", "mean"),
        avg_macro_f1_3=("macro_f1_3", "mean"),
        avg_acc_3=("acc_3", "mean"),
        avg_coverage=("coverage_vs_gold", "mean"),
    ).sort_values(["avg_qwk_5", "avg_macro_f1_3"], ascending=[False, False])

    model_avg.to_csv(reports_path / "model_average_scores.csv", index=False, encoding="utf-8-sig")

    # 4) One combined figure for all metric averages
    metric_plot_config = [
        ("avg_qwk_5", "Average QWK (5-class)", False),
        ("avg_mae_5", "Average MAE (5-class)", True),
        ("avg_acc_5", "Average Accuracy (5-class)", False),
        ("avg_macro_f1_5", "Average Macro F1 (5-class)", False),
        ("avg_macro_f1_3", "Average Macro F1 (3-class)", False),
        ("avg_acc_3", "Average Accuracy (3-class)", False),
        ("avg_coverage", "Average Coverage", False),
    ]

    fig, axes = plt.subplots(3, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, (metric_col, title, ascending) in enumerate(metric_plot_config):
        ax = axes[i]
        chart_df = model_avg.sort_values(metric_col, ascending=ascending)
        ax.bar(chart_df["model"], chart_df[metric_col])
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=35)

    # Hide unused subplots
    for j in range(len(metric_plot_config), len(axes)):
        axes[j].axis("off")

    fig.tight_layout()
    fig.savefig(reports_path / "all_model_metrics_grid.png", dpi=200)
    plt.close(fig)

    # 5) Individual chart files for each metric
    for metric_col, title, ascending in metric_plot_config:
        plt.figure(figsize=(10, 5))
        chart_df = model_avg.sort_values(metric_col, ascending=ascending)
        plt.bar(chart_df["model"], chart_df[metric_col])
        plt.xticks(rotation=35, ha="right")
        plt.ylabel(title)
        plt.title(title + " by Model")
        plt.tight_layout()
        plt.savefig(reports_path / f"{metric_col}_by_model.png", dpi=200)
        plt.close()

    # 6) Scatter plot for error-quality tradeoff
    plt.figure(figsize=(8, 6))
    plt.scatter(df["mae_5"], df["macro_f1_3"])
    plt.xlabel("MAE (5-class, lower is better)")
    plt.ylabel("Macro F1 (3-class, higher is better)")
    plt.title("MAE_5 vs Macro_F1_3 across dataset-model pairs")
    plt.tight_layout()
    plt.savefig(reports_path / "mae_5_vs_macro_f1_3.png", dpi=200)
    plt.close()

    print("Saved report folder:", reports_path)
    print("Files:")
    for p in sorted(reports_path.glob("*")):
        print("-", p.name)

    return slide_table, model_avg, pivot_qwk, pivot_f1_3, pivot_acc_5, pivot_acc_3


slide_table, model_avg, pivot_qwk, pivot_f1_3, pivot_acc_5, pivot_acc_3 = build_presentation_outputs(
    summary_csv="evaluation_summary.csv",
    reports_dir="evaluation_reports",
)

slide_table.head(15)

Saved report folder: evaluation_reports
Files:
- acc_3_pivot.csv
- acc_5_pivot.csv
- all_model_metrics_grid.png
- avg_acc_3_by_model.png
- avg_acc_5_by_model.png
- avg_coverage_by_model.png
- avg_macro_f1_3_by_model.png
- avg_macro_f1_5_by_model.png
- avg_mae_5_by_model.png
- avg_qwk_5_by_model.png
- macro_f1_3_pivot.csv
- mae_5_vs_macro_f1_3.png
- model_average_scores.csv
- qwk_5_pivot.csv
- slide_table.csv


,dataset,model,valid_rows,coverage_vs_gold,qwk_5,mae_5,acc_5,macro_f1_5,macro_f1_3,acc_3
0,apolitical_relgious_hefajot,Qwen 2.5 7b,17,0.9444,0.0000,1.2941,0.1765,0.0600,0.1000,0.1765
1,apolitical_relgious_hefajot,GPT-4o,16,0.8889,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
2,apolitical_relgious_hefajot,Claude Sonnet 5,18,1.0000,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
3,apolitical_relgious_hefajot,Gemini 3.1 Pro,18,1.0000,NaN,0.0000,1.0000,0.2000,0.3333,1.0000
4,apolitical_relgious_iskon,GPT-4o,8,1.0000,0.5000,0.2500,0.7500,0.3000,0.5000,0.7500
5,apolitical_relgious_iskon,Gemini 3.1 Pro,8,1.0000,0.0000,0.2500,0.7500,0.1714,0.2857,0.7500
6,apolitical_relgious_iskon,Claude Sonnet 5,8,1.0000,0.0000,0.3750,0.6250,0.1538,0.2564,0.6250
7,apolitical_relgious_iskon,Qwen 2.5 7b,7,0.8750,0.0000,1.7143,0.0000,0.0000,0.0952,0.1429
8,apolitical_religious_tablig,Claude Sonnet 5,9,1.0000,0.7429,0.3333,0.6667,0.3818,0.9030,0.8889
9,apolitical_religious_tablig,Gemini 3.1 Pro,9,1.0000,0.6087,0.2222,0.7778,0.3267,0.5444,0.7778


In [6]:
from pathlib import Path
import pandas as pd


def export_category_views(summary_csv="../evaluation_summary.csv", reports_dir="../evaluation_reports"):
    summary_path = Path(summary_csv)
    reports_path = Path(reports_dir)
    reports_path.mkdir(parents=True, exist_ok=True)

    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")

    df = pd.read_csv(summary_path)

    geo_df = df[df["dataset"].str.startswith("geo_", na=False)].copy()
    plt_df = df[df["dataset"].str.startswith("plt_", na=False)].copy()

    geo_df = geo_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True])
    plt_df = plt_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True])

    geo_df.to_csv(reports_path / "geo_summary.csv", index=False, encoding="utf-8-sig")
    plt_df.to_csv(reports_path / "plt_summary.csv", index=False, encoding="utf-8-sig")

    # Best model per dataset for quick slide/table
    best_geo = geo_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True]).groupby("dataset", as_index=False).first()
    best_plt = plt_df.sort_values(["dataset", "macro_f1", "mae"], ascending=[True, False, True]).groupby("dataset", as_index=False).first()

    best_geo.to_csv(reports_path / "best_geo_models.csv", index=False, encoding="utf-8-sig")
    best_plt.to_csv(reports_path / "best_plt_models.csv", index=False, encoding="utf-8-sig")

    print("Geo rows:", len(geo_df), "| Geo datasets:", geo_df["dataset"].nunique())
    print("Plt rows:", len(plt_df), "| Plt datasets:", plt_df["dataset"].nunique())
    print("Saved files:")
    print("-", reports_path / "geo_summary.csv")
    print("-", reports_path / "plt_summary.csv")
    print("-", reports_path / "best_geo_models.csv")
    print("-", reports_path / "best_plt_models.csv")

    return geo_df, plt_df, best_geo, best_plt


geo_df, plt_df, best_geo, best_plt = export_category_views(
    summary_csv="../evaluation_summary.csv",
    reports_dir="../evaluation_reports",
)

best_geo, best_plt

Geo rows: 14 | Geo datasets: 4
Plt rows: 8 | Plt datasets: 4
Saved files:
- ../evaluation_reports/geo_summary.csv
- ../evaluation_reports/plt_summary.csv
- ../evaluation_reports/best_geo_models.csv
- ../evaluation_reports/best_plt_models.csv


(                dataset   model                      gold_file  \
 0             geo_india  gemini      gold_standard_india.jsonl   
 1  geo_israel_palestine  gemini  gold_standard_palestine.jsonl   
 2          geo_pakistan  gemini   gold_standard_pakistan.jsonl   
 3              geo_west  gemini       gold_standard_west.jsonl   
 
                                pred_file  gold_rows  pred_rows_success  \
 0     gemini3.1_pro_extended_india.jsonl         50                 50   
 1      gemini3.1_pro_ext_palestine.jsonl         10                 10   
 2  gemini3.1_pro_extended_pakistan.jsonl         41                 43   
 3           gemini3.1_pro_ext_west.jsonl         24                 24   
 
    merged_rows  valid_rows  coverage_vs_gold     mae  accuracy  macro_f1  
 0           50          50               1.0  0.0400    0.9600    0.7613  
 1           10          10               1.0  0.0000    1.0000    0.6000  
 2           41          41               1.0  0.0732    0